In [ ]:
import pandas as pd
import polars as pl
from loguru import logger
from pathlib import Path
import hydra
from functools import reduce
from omegaconf import DictConfig, OmegaConf

import hvplot.pandas
import holoviews as hv
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

tqdm.pandas()

from workspace_dani.data_preprocessing.utils.data_handling import load_parquet_table

hv.extension("bokeh")

In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

### Small Cohort (Cohort B)
dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("sepsis-type-b-cohort") / Path(
    "2024-05-31") / Path("build_dataset_ts.parquet")



In [ ]:
df_dataset = load_parquet_table(parquet_path=dataset_path, verbose=False, debug_mode=False).head(n=1000000)


In [ ]:
df_dataset.head()

In [ ]:
sepsis_labels_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path(
    "sepsis_labels_concise.parquet")

df_sepsis_labels = load_parquet_table(parquet_path=sepsis_labels_path, verbose=False)


In [ ]:
df_sepsis_labels.head()

In [ ]:
patient_ids_with_sepsis = df_sepsis_labels["patientunitstayid"].unique().tolist()

In [ ]:

def add_sepsis_label_per_patient(patient_group: pd.DataFrame.groupby):
    patient_id = patient_group["patientunitstayid"].iloc[0]

    df_patient_group_to_label = patient_group.copy()
    df_patient_group_to_label["sepsis_diagnosis"] = pd.Series(dtype=bool)
    df_patient_group_to_label["sepsis_3"] = pd.Series(dtype=bool)
    df_patient_group_to_label["time_offset"] = pd.to_timedelta(df_patient_group_to_label["time_offset"], unit="min")

    if patient_id in patient_ids_with_sepsis:
        # patient has a  sepsis label

        sepsis_label = df_sepsis_labels[df_sepsis_labels["patientunitstayid"] == patient_id]
        sepsis_label = sepsis_label.rename(columns={"sepsis_offset": "time_offset"})
        
        # Todo: This might be redundant if I set the dtype to timedelta64[s] explicitly in the dataset build pipeline 
        sepsis_label["time_offset"] = sepsis_label["time_offset"].astype("timedelta64[ns]")

        sepsis_patient_offset = sepsis_label.loc[:, "time_offset"].values[0]

        patient_exact_sepsis_label_row = pd.DataFrame(
            data={"time_offset": [sepsis_patient_offset], "patientunitstayid": [patient_id]})

        df_patient_labeled = pd.merge(left=df_patient_group_to_label, right=patient_exact_sepsis_label_row,
                                      on=["patientunitstayid", "time_offset"], how="outer")

        df_patient_labeled.loc[df_patient_labeled["time_offset"] < sepsis_patient_offset, ["sepsis_3"]] = False
        df_patient_labeled.loc[df_patient_labeled["time_offset"] >= sepsis_patient_offset, ["sepsis_3"]] = True

        df_patient_labeled["sepsis_diagnosis"] = True

    else:
        # patient doesn't have a sepsis label
        df_patient_group_to_label["sepsis_diagnosis"] = False
        df_patient_labeled = df_patient_group_to_label.assign(sepsis_3=False)

    return df_patient_labeled


df_dataset_with_sepsis = df_dataset.groupby(by="patientunitstayid", group_keys=False).apply(
    add_sepsis_label_per_patient, include_groups=True)



In [ ]:
df_dataset_with_sepsis.head()

Taking a look at patient sample that get sepsis

In [ ]:
ids_sepsis_patients_cohort = df_dataset_with_sepsis[df_dataset_with_sepsis["sepsis_diagnosis"] == True][
    "patientunitstayid"].unique().tolist()


In [ ]:
df_sepsis_sample = df_dataset_with_sepsis.loc[
    df_dataset_with_sepsis["patientunitstayid"] == ids_sepsis_patients_cohort[2], ["time_offset", "sepsis_3", "sepsis_diagnosis"]]

In [ ]:

df_sepsis_sample_all_col = df_dataset_with_sepsis.loc[
    df_dataset_with_sepsis["patientunitstayid"] == ids_sepsis_patients_cohort[2]]


In [ ]:
offset_sepsis_sample = df_sepsis_labels.loc[
    df_sepsis_labels["patientunitstayid"] == ids_sepsis_patients_cohort[2], ["sepsis_offset"]]
offset_sepsis_sample

Sanity check on the subset of patients that doesn't get sepsis


In [ ]:
ids_no_sepsis_patients_cohort = df_dataset_with_sepsis[df_dataset_with_sepsis["sepsis_diagnosis"] == False][
    "patientunitstayid"].unique().tolist()


In [ ]:
df_patients_with_no_sepsis = df_dataset_with_sepsis.loc[df_dataset_with_sepsis["patientunitstayid"].isin(ids_no_sepsis_patients_cohort)]

In [ ]:
df_patients_with_no_sepsis.describe(include="all")

In [ ]:
df_patients_with_no_sepsis[["sepsis_diagnosis", "sepsis_3"]].value_counts()

### -> LGTM, no stray sepsis labels